# Zomato Bangalore Restaurants — Business Analytics (Review 1)
### Main Research Question: *What factors influence restaurant ratings and customer preferences?*
**Academic Level:** Undergraduate Business Analytics  
**Focus Areas:** Data Understanding & Cleaning, Exploratory Data Analysis, Presentation-Quality Visualizations, Predictive Modelling, Model Evaluation & Business Insights


## 1. Dataset Understanding
In this section, we load the raw dataset (`zomato.csv`), inspect its dimensions, examine column data types, and check for null values and duplicate records.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Set presentation styles
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['font.sans-serif'] = 'Arial'

# Load Dataset
df = pd.read_csv('zomato.csv')
print("Dataset Shape:", df.shape)
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])
df.head(3)


In [ ]:
# Check data types and missing values
print("Column Names & Data Types:")
print(df.dtypes)

print("\nMissing Values Count:")
print(df.isnull().sum())

print("\nExact Duplicate Records (all 17 columns):", df.duplicated().sum())


## 2. Data Cleaning & Preprocessing
To prepare the dataset for business analysis and machine learning:
1. **Drop non-analytical text/metadata columns**: `url`, `phone`, `reviews_list`, `menu_item`, and unstructured `address`.
2. **Remove exact duplicates**: Deduplicate records across the retained business attributes.
3. **Parse `rate`**: Extract numerical rating from the string format (`'4.1/5'` -> `4.1`), treat `'NEW'` and `'-'` as unrated (`NaN`), and drop unrated records.
4. **Clean `approx_cost(for two people)`**: Remove thousands separator commas (`'1,200'` -> `1200`) and cast to numeric float.
5. **Ensure `votes` is integer**: Clean and cast to integer.
6. **Binary Encoding**: Map `online_order` and `book_table` from `'Yes'/'No'` to `1/0`.


In [ ]:
# 1. Drop irrelevant columns
drop_cols = ['url', 'phone', 'reviews_list', 'menu_item', 'address']
df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 2. Deduplicate
df_clean = df_clean.drop_duplicates()

# 3. Clean 'rate'
def parse_rate(val):
    if pd.isna(val) or str(val).strip() in ['NEW', '-', '']:
        return np.nan
    try:
        return float(str(val).split('/')[0].strip())
    except:
        return np.nan

df_clean['rate'] = df_clean['rate'].apply(parse_rate)

# 4. Clean 'approx_cost(for two people)'
cost_col = 'approx_cost(for two people)'
df_clean['approx_cost'] = df_clean[cost_col].apply(lambda x: float(str(x).replace(',', '').strip()) if pd.notna(x) else np.nan)
df_clean.drop(columns=[cost_col], inplace=True)

# 5. Clean votes
df_clean['votes'] = pd.to_numeric(df_clean['votes'], errors='coerce').fillna(0).astype(int)

# 6. Encode binary flags
df_clean['online_order_encoded'] = df_clean['online_order'].map({'Yes': 1, 'No': 0})
df_clean['book_table_encoded'] = df_clean['book_table'].map({'Yes': 1, 'No': 0})

# 7. Drop rows missing target rate & key features
df_valid = df_clean.dropna(subset=['rate', 'approx_cost', 'rest_type', 'cuisines', 'location']).copy()

# Feature extraction: Primary restaurant type and primary cuisine
df_valid['primary_rest_type'] = df_valid['rest_type'].apply(lambda x: str(x).split(',')[0].strip())
df_valid['primary_cuisine'] = df_valid['cuisines'].apply(lambda x: str(x).split(',')[0].strip())

print("Cleaned and validated rows:", len(df_valid))
df_valid[['name', 'rate', 'approx_cost', 'votes', 'online_order_encoded', 'book_table_encoded']].head()


## 3. Exploratory Data Analysis (EDA)
Here we compute statistical summaries (mean, median, min, max, std) for ratings, cost, and customer votes, along with group-level performance benchmarks.


In [ ]:
# Summary statistics
print("=== DESCRIPTIVE STATISTICS ===")
print(df_valid[['rate', 'approx_cost', 'votes']].describe().T)

print("\n=== ONLINE ORDER IMPACT ON RATING ===")
print(df_valid.groupby('online_order')['rate'].agg(['count', 'mean', 'median', 'std']))

print("\n=== TABLE BOOKING IMPACT ON RATING ===")
print(df_valid.groupby('book_table')['rate'].agg(['count', 'mean', 'median', 'std']))

print("\n=== CORRELATIONS WITH RATING ===")
print(df_valid[['rate', 'votes', 'approx_cost', 'online_order_encoded', 'book_table_encoded']].corr()['rate'])


## 4. Visualizations & Analytical Interpretations
We construct six presentation-quality charts answering key business questions.


In [ ]:
# 4.1 Distribution of Restaurant Ratings
plt.figure(figsize=(9, 5))
sns.histplot(df_valid['rate'], bins=31, kde=True, color='#E23744', edgecolor='white')
plt.axvline(df_valid['rate'].mean(), color='#1A237E', linestyle='--', linewidth=2, label=f"Mean: {df_valid['rate'].mean():.2f}")
plt.axvline(df_valid['rate'].median(), color='#004D40', linestyle=':', linewidth=2, label=f"Median: {df_valid['rate'].median():.2f}")
plt.title("Figure 1: Distribution of Restaurant Ratings in Bangalore", fontsize=13, fontweight='bold')
plt.xlabel("Rating (out of 5.0)")
plt.ylabel("Number of Restaurants")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 4.2 Rating vs Number of Votes (Engagement Tiers)
bins_votes = [0, 50, 150, 500, 1000, 5000, 20000]
labels_votes = ['<50 (Low)', '50-150', '150-500 (Mod)', '500-1k (High)', '1k-5k (V.High)', '5k+ (Mega)']
df_valid['vote_bracket'] = pd.cut(df_valid['votes'], bins=bins_votes, labels=labels_votes, include_lowest=True)
vote_stats = df_valid.groupby('vote_bracket', observed=True)['rate'].mean().reset_index()

plt.figure(figsize=(9, 5))
ax = sns.barplot(x='vote_bracket', y='rate', data=vote_stats, hue='vote_bracket', palette='Blues_r', legend=False)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2., p.get_height() - 0.25),
                ha='center', va='center', color='white', fontweight='bold')
plt.title("Figure 2: Customer Engagement (Votes) vs Average Rating", fontsize=13, fontweight='bold')
plt.xlabel("Engagement Bracket (Votes)")
plt.ylabel("Average Rating")
plt.ylim(3.0, 4.6)
plt.tight_layout()
plt.show()


In [ ]:
# 4.3 Rating vs Approximate Cost for Two
bins_cost = [0, 300, 600, 1000, 2000, 7000]
labels_cost = ['Budget (<300)', 'Mid-Low (300-600)', 'Mid-High (600-1k)', 'Premium (1k-2k)', 'Fine Dining (>2k)']
df_valid['cost_bracket'] = pd.cut(df_valid['approx_cost'], bins=bins_cost, labels=labels_cost, include_lowest=True)
cost_stats = df_valid.groupby('cost_bracket', observed=True)['rate'].mean().reset_index()

plt.figure(figsize=(9, 5))
ax = sns.barplot(x='cost_bracket', y='rate', data=cost_stats, hue='cost_bracket', palette='YlOrRd', legend=False)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2., p.get_height() - 0.25),
                ha='center', va='center', color='white', fontweight='bold')
plt.title("Figure 3: Approximate Cost for Two vs Average Rating", fontsize=13, fontweight='bold')
plt.xlabel("Cost Bracket (INR)")
plt.ylabel("Average Rating")
plt.ylim(3.0, 4.6)
plt.tight_layout()
plt.show()


In [ ]:
# 4.4 Average Rating by Restaurant Type (Top 12)
top_types = df_valid.groupby('primary_rest_type')['rate'].agg(['count', 'mean']).sort_values(by='count', ascending=False).head(12)
top_types = top_types.sort_values(by='mean', ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(top_types.index, top_types['mean'], color='#1C2938')
for p in bars:
    plt.text(p.get_width() + 0.03, p.get_y() + p.get_height()/2, f"{p.get_width():.2f}", va='center', fontweight='bold')
plt.axvline(df_valid['rate'].mean(), color='#E23744', linestyle='--', label=f"City Avg: {df_valid['rate'].mean():.2f}")
plt.title("Figure 4: Average Restaurant Rating by Format / Dining Style", fontsize=13, fontweight='bold')
plt.xlabel("Average Rating")
plt.xlim(3.0, 4.4)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 4.5 Average Rating by Top 15 Locations
top_locs = df_valid.groupby('location')['rate'].agg(['count', 'mean']).sort_values(by='count', ascending=False).head(15)
top_locs = top_locs.sort_values(by='mean', ascending=True)

plt.figure(figsize=(10, 6.5))
bars = plt.barh(top_locs.index, top_locs['mean'], color='#2E7D32')
for p in bars:
    plt.text(p.get_width() + 0.03, p.get_y() + p.get_height()/2, f"{p.get_width():.2f}", va='center', fontweight='bold')
plt.axvline(df_valid['rate'].mean(), color='#E23744', linestyle='--', label=f"City Avg: {df_valid['rate'].mean():.2f}")
plt.title("Figure 5: Performance Benchmark across Top 15 Bangalore Hubs", fontsize=13, fontweight='bold')
plt.xlabel("Average Rating")
plt.xlim(3.0, 4.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 4.6 Online Ordering Availability vs Rating
plt.figure(figsize=(7, 5))
sns.boxplot(x='online_order', y='rate', data=df_valid, hue='online_order', palette=['#78909C', '#E23744'], width=0.4, legend=False)
plt.title("Figure 6: Online Ordering Availability vs Rating", fontsize=13, fontweight='bold')
plt.xlabel("Online Ordering Enabled")
plt.ylabel("Rating (out of 5.0)")
plt.tight_layout()
plt.show()


## 5. Business Analysis & Hypotheses Testing
1. **Engagement Impact**: Customer votes exhibit a strong positive correlation (+0.435) with ratings.
2. **Pricing Impact**: Cost for two has a moderate-to-strong correlation (+0.385) with ratings.
3. **Format Superiority**: Experiential dining formats (Pubs, Microbreweries, Cafes) outperform high-turnover Quick Bites by ~0.55 rating points.
4. **Location Clustering**: Foodie corridors (Koramangala 5th Block: 4.01, Indiranagar: 3.83) outperform student/budget clusters (BTM: 3.57).
5. **Online Ordering Lift**: Statistically significant rating lift (+0.064, p < 1e-40).
6. **Table Booking Lift**: Premium feature with massive rating difference (+0.521, p = 0.0).


In [ ]:
# Statistical Significance Tests
tt_online = stats.ttest_ind(
    df_valid[df_valid['online_order'] == 'Yes']['rate'],
    df_valid[df_valid['online_order'] == 'No']['rate'],
    equal_var=False
)
print(f"Online Order Two-Sample T-Test: t = {tt_online.statistic:.4f}, p-value = {tt_online.pvalue:.4e}")

tt_table = stats.ttest_ind(
    df_valid[df_valid['book_table'] == 'Yes']['rate'],
    df_valid[df_valid['book_table'] == 'No']['rate'],
    equal_var=False
)
print(f"Table Booking Two-Sample T-Test: t = {tt_table.statistic:.4f}, p-value = {tt_table.pvalue:.4e}")


## 6. Predictive Modelling
We build and compare two predictive models to forecast restaurant ratings:
1. **Linear Regression** (Interpretable linear baseline)
2. **Random Forest Regressor** (Ensemble tree-based model capturing non-linear feature interactions)

Features used: `votes`, `approx_cost`, `online_order`, `book_table`, `location`, `rest_type`, `cuisine`.


In [ ]:
# Category simplification to manage cardinality
top_loc_list = df_valid['location'].value_counts().nlargest(20).index
df_valid['location_clean'] = df_valid['location'].apply(lambda x: x if x in top_loc_list else 'Other')

top_cui_list = df_valid['primary_cuisine'].value_counts().nlargest(15).index
df_valid['cuisine_clean'] = df_valid['primary_cuisine'].apply(lambda x: x if x in top_cui_list else 'Other')

top_type_list = df_valid['primary_rest_type'].value_counts().nlargest(10).index
df_valid['rest_type_clean'] = df_valid['primary_rest_type'].apply(lambda x: x if x in top_type_list else 'Other')

features = ['votes', 'approx_cost', 'online_order_encoded', 'book_table_encoded', 'location_clean', 'rest_type_clean', 'cuisine_clean']
X = df_valid[features]
y = df_valid['rate']

# 80-20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

cat_features = ['location_clean', 'rest_type_clean', 'cuisine_clean']
num_features = ['votes', 'approx_cost', 'online_order_encoded', 'book_table_encoded']

ct = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
    ]
)

X_train_trans = ct.fit_transform(X_train)
X_test_trans = ct.transform(X_test)
cat_names = ct.named_transformers_['cat'].get_feature_names_out(cat_features)
all_feature_names = num_features + list(cat_names)

# Train Linear Regression
lr = LinearRegression()
lr.fit(X_train_trans, y_train)
y_pred_lr = lr.predict(X_test_trans)

# Train Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, max_depth=16, min_samples_split=5, random_state=42, n_jobs=-1)
rf.fit(X_train_trans, y_train)
y_pred_rf = rf.predict(X_test_trans)

# Evaluation
eval_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regressor'],
    'MAE': [mean_absolute_error(y_test, y_pred_lr), mean_absolute_error(y_test, y_pred_rf)],
    'RMSE': [np.sqrt(mean_squared_error(y_test, y_pred_lr)), np.sqrt(mean_squared_error(y_test, y_pred_rf))],
    'R² Score': [r2_score(y_test, y_pred_lr), r2_score(y_test, y_pred_rf)]
})
eval_df


## 7. Model Analysis & Feature Importance
We extract and analyze the feature importances from the Random Forest model to understand what factors most heavily influence restaurant ratings.


In [ ]:
# Group importances by domain category
group_imp = {
    'Customer Engagement (Votes)': rf.feature_importances_[0],
    'Location (Geography)': sum([rf.feature_importances_[i] for i, name in enumerate(all_feature_names) if 'location' in name]),
    'Price Level (Approx Cost)': rf.feature_importances_[1],
    'Cuisine Offering': sum([rf.feature_importances_[i] for i, name in enumerate(all_feature_names) if 'cuisine' in name]),
    'Dining Format (Rest Type)': sum([rf.feature_importances_[i] for i, name in enumerate(all_feature_names) if 'rest_type' in name]),
    'Table Booking Support': rf.feature_importances_[3],
    'Online Ordering Enabled': rf.feature_importances_[2],
}
group_imp_series = pd.Series(group_imp).sort_values(ascending=True)

plt.figure(figsize=(10, 5.5))
bars = plt.barh(group_imp_series.index, group_imp_series.values * 100, color='#E23744')
for p in bars:
    plt.text(p.get_width() + 1.0, p.get_y() + p.get_height()/2, f"{p.get_width():.1f}%", va='center', fontweight='bold')
plt.title("Figure 7: Domain Feature Importance in Predicting Ratings (Random Forest)", fontsize=13, fontweight='bold')
plt.xlabel("Relative Importance (%)")
plt.xlim(0, 75)
plt.tight_layout()
plt.show()


## 8. Final Business Insights
Here are the core findings formatted as **Finding → Evidence → Business Implication**:

1. **Customer Engagement is the Primary Driver of High Ratings**
   - *Finding*: Number of customer votes is the single largest determinant of rating.
   - *Evidence*: `votes` carries +0.435 correlation and accounts for 63.6% of Random Forest feature importance. Restaurants with >1,000 votes average 4.22 vs 3.49 for <50 votes.
   - *Implication*: Encouraging satisfied diners to review on Zomato creates a self-reinforcing flywheel of visibility, credibility, and higher ratings.

2. **Table Booking Capability is a Strong Premium Signal**
   - *Finding*: Supporting table reservations is associated with superior perceived quality.
   - *Evidence*: Table booking restaurants average 4.14 vs 3.62 for non-booking (+0.52 point lift, t = 118.16, p = 0.0).
   - *Implication*: Even casual dining spots can unlock premium customer perception and higher dwell times by enabling digital reservation infrastructure.

3. **Pricing Power Reflects Experience Quality**
   - *Finding*: Higher cost for two consistently correlates with superior customer ratings.
   - *Evidence*: Positive correlation (+0.385); Fine Dining (>Rs. 2,000) averages 4.24 vs Budget (<Rs. 300) at 3.46.
   - *Implication*: Competing solely on rock-bottom price in Bangalore hurts perceived quality. Mid-to-premium positioning allows investment in food consistency and ambiance.

4. **Experiential Formats Trump Quick Turnover**
   - *Finding*: Experiential dining formats (Pubs, Microbreweries, Cafes) outperform fast-service formats.
   - *Evidence*: Pubs (4.10) and Cafes (3.87) consistently outperform Quick Bites (3.55) and Takeaway (3.51).
   - *Implication*: Bangalore consumers reward ambiance, curated beverage programs, and social experience over transactional speed.

5. **Strategic Location Clustering Dictates Demand Quality**
   - *Finding*: Established foodie hubs command significantly higher ratings and customer engagement than emerging suburban corridors.
   - *Evidence*: Koramangala 5th Block averages 4.01 rating and 965 avg votes; Indiranagar averages 3.83 rating. In contrast, BTM averages 3.57 rating and 148 votes despite higher store count.
   - *Implication*: High footfall density in BTM is budget-constrained; premium concepts must target Koramangala or Indiranagar to achieve high customer sentiment.


## 9. Presentation Deck Structure (10-12 Slides for Review 1)
- **Slide 1**: Title & Executive Context
- **Slide 2**: The Business Problem & Objectives
- **Slide 3**: Dataset Architecture & Profiling
- **Slide 4**: Data Cleaning & Preprocessing Decisions
- **Slide 5**: Exploratory Data Analysis & Rating Distribution (Figure 1)
- **Slide 6**: Customer Engagement & Social Proof Analysis (Figure 2)
- **Slide 7**: Pricing Power & Cost Bracket Analysis (Figure 3)
- **Slide 8**: Format & Location Benchmark (Figures 4 & 5)
- **Slide 9**: Operational Digital Capabilities (Figure 6 & T-tests)
- **Slide 10**: Machine Learning Modelling & Pipeline (LR vs RF)
- **Slide 11**: Model Evaluation & Feature Importance (Figures 7 & 8)
- **Slide 12**: Actionable Strategic Recommendations & Conclusion
